In [30]:
try:
    %reload_ext autoreload
except:
    %load_ext autoreload
%autoreload 2
from tqdm import tqdm
# Basic useful imports
import re
import time
import yaml
from pprint import pprint
from pathlib import Path
import h5py
from copy import deepcopy
from typing import Dict

# Data manipulation
import numpy as np
from scipy.special import erf
from scipy.integrate import quad, solve_ivp

from IPython.display import HTML

# Visualization
import matplotlib.pyplot as plt

from foxlink.gen_fast_gaussian_moment import (compute_Ik_l_all, Params)
from foxlink.DW_gauss_integration import gaussian_rect_integrals_up_to2_numba_dw

from numba import njit
from foxlink.me_zrl_helpers import fast_zrl_src_kl

data_path = Path("../../examples/new_me_test/NFIlMomentExpansionSolver.h5")
data_path.exists()


True

In [2]:
graph_sty = {
    "axes.titlesize": 20,
    "axes.labelsize": 24,
    "lines.linewidth": 2,
    "lines.markersize": 10,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "font.size": 20,
    # "font.sans-serif": 'Helvetica',
    "text.usetex": False,
    'mathtext.fontset': 'cm',
}
plt.style.use(graph_sty)

In [3]:
import numpy as np
from dataclasses import dataclass
from typing import Dict
from scipy.stats import norm, multivariate_normal


# --- mapped-parameter evaluator ---
@dataclass
class ParamsMapped:
    a: float      # >0
    q: float      # = r^2 >= 0
    a_ij: float
    a_ji: float
    d: float      # |d|<1
    L_i: float    # >0
    L_j: float    # >0

def compute_Ik_l_all_mapped(p: ParamsMapped) -> Dict[str, float]:
    # --- special functions and required partials of Φ2 ---
    def phi(z): return norm.pdf(z)
    def Phi(z): return norm.cdf(z)

    def phi2(x, y, rho):
        denom = 2*np.pi*np.sqrt(1 - rho**2)
        q = (x**2 - 2*rho*x*y + y**2) / (2*(1 - rho**2))
        return np.exp(-q) / denom

    def Phi2(x, y, rho):
        return multivariate_normal(mean=[0.0,0.0], cov=[[1.0,rho],[rho,1.0]]).cdf([x,y])

    def dPhi2_dx(x, y, rho):
        return phi(x) * Phi((y - rho*x)/np.sqrt(1 - rho**2))

    def dPhi2_dy(x, y, rho):
        return phi(y) * Phi((x - rho*y)/np.sqrt(1 - rho**2))

    def d2Phi2_dxx(x, y, rho):
        return -x*phi(x)*Phi((y - rho*x)/np.sqrt(1 - rho**2)) \
            - (rho/np.sqrt(1 - rho**2))*phi2(x,y,rho)

    def d2Phi2_dyy(x, y, rho):
        return -y*phi(y)*Phi((x - rho*y)/np.sqrt(1 - rho**2)) \
            - (rho/np.sqrt(1 - rho**2))*phi2(x,y,rho)

    def d2Phi2_dxy(x, y, rho):
        return phi2(x, y, rho)

    def inclusion_exclusion(func, zi_plus, zi_minus, zj_plus, zj_minus):
        return (func(zi_plus,  zj_plus)
            - func(zi_minus, zj_plus)
            - func(zi_plus,  zj_minus)
            + func(zi_minus, zj_minus))

    a, q, aij, aji, d, L_i, L_j = p.a, p.q, p.a_ij, p.a_ji, p.d, p.L_i, p.L_j
    if not (a > 0 and L_i > 0 and L_j > 0 and abs(d) < 1 and q >= 0):
        raise ValueError("Need a>0, L_i>0, L_j>0, |d|<1, q>=0.")

    one_minus_d2 = 1 - d**2
    rho = d
    sigma = np.sqrt(1.0/(2*a*one_minus_d2))

    # mean shift and standardized bounds
    mu_i = (aij + d*aji)/(a*one_minus_d2)
    mu_j = (d*aij + aji)/(a*one_minus_d2)

    zi_plus  = (+L_i/2 - mu_i)/sigma
    zi_minus = (-L_i/2 - mu_i)/sigma
    zj_plus  = (+L_j/2 - mu_j)/sigma
    zj_minus = (-L_j/2 - mu_j)/sigma

    # rectangle mass
    Delta = inclusion_exclusion(lambda x,y: Phi2(x,y,rho),
                                zi_plus, zi_minus, zj_plus, zj_minus)

    # prefactor
    expo = -a*q + (aij**2 + 2*d*aij*aji + aji**2)/(a*one_minus_d2)
    P = (np.pi/(a*np.sqrt(one_minus_d2))) * np.exp(expo)

    I00 = P * Delta

    # derivatives of log P wrt aij, aji
    Eaij = (2*(aij + d*aji)) / (a*one_minus_d2)
    Eaji = (2*(aji + d*aij)) / (a*one_minus_d2)

    # their second derivatives (constants)
    Eaij_aij = 2/(a*one_minus_d2)
    Eaji_aji = 2/(a*one_minus_d2)
    Eaij_aji = 2*d/(a*one_minus_d2)   # = Eaji_aij

    # how bounds move with aij, aji  (note the minus signs)
    base = np.sqrt(2*a/(one_minus_d2)) / a
    alpha_ib = - base
    alpha_jb = - d*base
    alpha_ic = - d*base
    alpha_jc = - base

    Sb = inclusion_exclusion(
        lambda x,y: alpha_ib*dPhi2_dx(x,y,rho) + alpha_jb*dPhi2_dy(x,y,rho),
        zi_plus, zi_minus, zj_plus, zj_minus
    )
    Sc = inclusion_exclusion(
        lambda x,y: alpha_ic*dPhi2_dx(x,y,rho) + alpha_jc*dPhi2_dy(x,y,rho),
        zi_plus, zi_minus, zj_plus, zj_minus
    )

    # scale from identity: I_{1,0} = (1/(2a)) ∂_{aij} I00, etc.
    scale = 1.0/(2*a)

    I10 = scale * (Eaij*I00 + P*Sb)
    I01 = scale * (Eaji*I00 + P*Sc)

    Sbb = inclusion_exclusion(
        lambda x,y: (alpha_ib**2)*d2Phi2_dxx(x,y,rho)
                  + 2*alpha_ib*alpha_jb*d2Phi2_dxy(x,y,rho)
                  + (alpha_jb**2)*d2Phi2_dyy(x,y,rho),
        zi_plus, zi_minus, zj_plus, zj_minus
    )
    Scc = inclusion_exclusion(
        lambda x,y: (alpha_ic**2)*d2Phi2_dxx(x,y,rho)
                  + 2*alpha_ic*alpha_jc*d2Phi2_dxy(x,y,rho)
                  + (alpha_jc**2)*d2Phi2_dyy(x,y,rho),
        zi_plus, zi_minus, zj_plus, zj_minus
    )
    Sbc = inclusion_exclusion(
        lambda x,y: (alpha_ib*alpha_ic)*d2Phi2_dxx(x,y,rho)
                  + (alpha_ib*alpha_jc + alpha_jb*alpha_ic)*d2Phi2_dxy(x,y,rho)
                  + (alpha_jb*alpha_jc)*d2Phi2_dyy(x,y,rho),
        zi_plus, zi_minus, zj_plus, zj_minus
    )

    I20 = (scale**2) * ((Eaij**2 + Eaij_aij)*I00 + 2*Eaij*P*Sb + P*Sbb)
    I02 = (scale**2) * ((Eaji**2 + Eaji_aji)*I00 + 2*Eaji*P*Sc + P*Scc)
    I11 = (scale**2) * ((Eaij*Eaji + Eaij_aji)*I00 + Eaij*P*Sc + Eaji*P*Sb + P*Sbc)

    return {"I00": float(I00), "I10": float(I10), "I01": float(I01),
            "I20": float(I20), "I02": float(I02), "I11": float(I11)}

# --- quick check example (should match old code under mapping) ---
if __name__ == "__main__":
    # pick original (r,b,c) and map to (q, a_ij, a_ji)
    a = 1.7
    r = 0.9
    b = 0.2
    c = -0.4
    d = 0.3
    L_i, L_j = 3.0, 2.5

    q   = r**2
    aij = -r*b
    aji = -r*c

    vals = compute_Ik_l_all_mapped(ParamsMapped(a=a, q=q, a_ij=aij, a_ji=aji, d=d, L_i=L_i, L_j=L_j))
    for k,v in vals.items():
        print(k, v)


I00 0.505326697567692
I10 -0.01571225225688273
I01 0.04921719183291847
I20 0.05255107888931717
I02 0.05078623952727816
I11 0.011465441584734896


In [4]:
import numpy as np
from math import sqrt, pi, exp
from scipy.special import ndtr          # Φ

# --- standard normal ---
def Phi(x): return ndtr(x)

def phi(x): return (1.0 / sqrt(2.0 * pi)) * np.exp(-0.5 * x * x)

# --- bivariate standard normal PDF ---
def phi2(x, y, rho):
    t = 1.0 - rho * rho
    return (1.0 / (2.0 * pi * sqrt(t))) * np.exp(-(x*x - 2.0*rho*x*y + y*y) / (2.0 * t))

# -- helper: all rectangle terms (mass, edge terms, explicit “derivatives” via PDFs)
def _rect_terms(u_minus, u_plus, v_minus, v_plus, rho):
    tau = sqrt(1.0 - rho * rho)
    mvn = multivariate_normal(mean=[0.0, 0.0], cov=[[1.0, rho], [rho, 1.0]])

    # rectangle probability P (4 CDFs)
    P_pp = float(mvn.cdf([u_plus,  v_plus ]))
    P_mp = float(mvn.cdf([u_minus, v_plus ]))
    P_pm = float(mvn.cdf([u_plus,  v_minus]))
    P_mm = float(mvn.cdf([u_minus, v_minus]))
    P = P_pp - P_mp - P_pm + P_mm

    # edge helpers
    def L_of_u(u): return Phi((v_plus  - rho*u) / tau) - Phi((v_minus - rho*u) / tau)
    def R_of_v(v): return Phi((u_plus  - rho*v) / tau) - Phi((u_minus - rho*v) / tau)

    # S_u, S_v
    L_up, L_um = L_of_u(u_plus), L_of_u(u_minus)
    R_vp, R_vm = R_of_v(v_plus), R_of_v(v_minus)
    s_u_plus, s_u_minus = phi(u_plus),  phi(u_minus)
    s_v_plus, s_v_minus = phi(v_plus),  phi(v_minus)
    Su = s_u_plus * L_up - s_u_minus * L_um
    Sv = s_v_plus * R_vp - s_v_minus * R_vm

    # corner PDFs
    c_pp = phi2(u_plus,  v_plus,  rho)
    c_pm = phi2(u_plus,  v_minus, rho)
    c_mp = phi2(u_minus, v_plus,  rho)
    c_mm = phi2(u_minus, v_minus, rho)

    # explicit “derivatives” (no numerical diffs)
    Du_plus, Du_minus = c_pp - c_pm, c_mp - c_mm
    Dv_plus, Dv_minus = c_pp - c_mp, c_pm - c_mm

    dSu_du = (-u_plus  * s_u_plus  * L_up  - rho * Du_plus) \
           + ( u_minus * s_u_minus * L_um  + rho * Du_minus)

    dSv_dv = (-v_plus  * s_v_plus  * R_vp  - rho * Dv_plus) \
           + ( v_minus * s_v_minus * R_vm  + rho * Dv_minus)

    dSu_dv =  c_pp - c_pm - c_mp + c_mm

    return P, Su, Sv, dSu_du, dSv_dv, dSu_dv

def gaussian_rect_integrals_up_to2_compact(alpha, q, b, a_ij, a_ji, L_i, L_j):
    """
    I_{k,l} on [-L_i/2,L_i/2]×[-L_j/2,L_j/2] for k+l <= 2.
    Assumes α>0, |b|<1, finite L_i,L_j.
    Returns dict {(k,l): value}.
    """
    rho = b
    denom = 1.0 - rho * rho
    inv_denom = 1.0 / denom

    # μ and Σ pieces
    mu_i = (a_ij + rho * a_ji) * inv_denom
    mu_j = (rho * a_ij + a_ji) * inv_denom
    sigma2 = inv_denom / (2.0 * alpha)
    sigma  = sqrt(sigma2)

    # standardized bounds
    half_i, half_j = 0.5 * L_i, 0.5 * L_j
    u_minus, u_plus = (-half_i - mu_i) / sigma, (half_i - mu_i) / sigma
    v_minus, v_plus = (-half_j - mu_j) / sigma, (half_j - mu_j) / sigma

    # rectangle terms
    P, Su, Sv, dSu_du, dSv_dv, dSu_dv = _rect_terms(u_minus, u_plus, v_minus, v_plus, rho)

    # global K
    a_dot_mu = a_ij * mu_i + a_ji * mu_j
    K = (pi / (alpha * sqrt(denom))) * exp(-alpha * (q - a_dot_mu))

    # assemble
    Su_rSv = Su + rho * Sv
    Sv_rSu = Sv + rho * Su

    I00 = K * P
    I10 = K * (mu_i * P - sigma * Su_rSv)
    I01 = K * (mu_j * P - sigma * Sv_rSu)
    I20 = K * ((mu_i * mu_i + sigma2) * P
               - 2.0 * mu_i * sigma * Su_rSv
               + sigma2 * (dSu_du + 2.0 * rho * dSu_dv + rho * rho * dSv_dv))
    I02 = K * ((mu_j * mu_j + sigma2) * P
               - 2.0 * mu_j * sigma * Sv_rSu
               + sigma2 * (dSv_dv + 2.0 * rho * dSu_dv + rho * rho * dSu_du))
    # FIX: add +rho*sigma2*P (the Σ_ij term)
    I11 = K * ((mu_i * mu_j + rho * sigma2) * P
               - mu_j * sigma * Su_rSv
               - mu_i * sigma * Sv_rSu
               + sigma2 * (rho * dSu_du + (1.0 + rho * rho) * dSu_dv + rho * dSv_dv))

    return {"I00": I00, "I10": I10, "I01": I01, "I11":I11 , "I20": I20, "I02": I02, }

In [12]:
from math import sqrt, pi, exp, erf
import math
from numba import njit
from scipy.stats import mvn  # fast Fortran-backed rectangular CDF (Φ₂ over a box)

# ---------- JIT-friendly scalar special functions ----------
SQRT1_2 = 1.0 / math.sqrt(2.0)
INV_2PI = 1.0 / (2.0 * math.pi)

@njit(fastmath=True)
def Phi_u(x):
    return 0.5 * (1.0 + math.erf(x * SQRT1_2))

@njit(fastmath=True)
def phi_u(x):
    return INV_2PI * math.sqrt(2.0 * math.pi) * math.exp(-0.5 * x * x)  # = 1/sqrt(2π)*exp(-x^2/2)

@njit(fastmath=True)
def phi2_u(x, y, rho):
    t = 1.0 - rho * rho
    z = (x * x - 2.0 * rho * x * y + y * y) / (2.0 * t)
    return INV_2PI / math.sqrt(t) * math.exp(-z)

# ---------- Core rectangle assembly (everything but P) ----------
@njit(fastmath=True)
def _assemble_given_P(mu_i, mu_j, sigma, sigma2, rho,
                      u_minus, u_plus, v_minus, v_plus, P):
    # L(u), R(v)
    tau = math.sqrt(1.0 - rho * rho)

    L_up = Phi_u((v_plus  - rho * u_plus ) / tau) - Phi_u((v_minus - rho * u_plus ) / tau)
    L_um = Phi_u((v_plus  - rho * u_minus) / tau) - Phi_u((v_minus - rho * u_minus) / tau)
    R_vp = Phi_u((u_plus  - rho * v_plus ) / tau) - Phi_u((u_minus - rho * v_plus ) / tau)
    R_vm = Phi_u((u_plus  - rho * v_minus) / tau) - Phi_u((u_minus - rho * v_minus) / tau)

    # S_u, S_v
    s_u_plus, s_u_minus = phi_u(u_plus), phi_u(u_minus)
    s_v_plus, s_v_minus = phi_u(v_plus), phi_u(v_minus)
    Su = s_u_plus * L_up - s_u_minus * L_um
    Sv = s_v_plus * R_vp - s_v_minus * R_vm

    # corner PDFs
    c_pp = phi2_u(u_plus,  v_plus,  rho)
    c_pm = phi2_u(u_plus,  v_minus, rho)
    c_mp = phi2_u(u_minus, v_plus,  rho)
    c_mm = phi2_u(u_minus, v_minus, rho)

    # pre-assembled combos
    Du_plus, Du_minus = c_pp - c_pm, c_mp - c_mm
    Dv_plus, Dv_minus = c_pp - c_mp, c_pm - c_mm

    dSu_du = (-u_plus  * s_u_plus  * L_up  - rho * Du_plus) \
           + ( u_minus * s_u_minus * L_um  + rho * Du_minus)

    dSv_dv = (-v_plus  * s_v_plus  * R_vp  - rho * Dv_plus) \
           + ( v_minus * s_v_minus * R_vm  + rho * Dv_minus)

    dSu_dv =  c_pp - c_pm - c_mp + c_mm

    # combine
    Su_rSv = Su + rho * Sv
    Sv_rSu = Sv + rho * Su

    # brackets (without global K)
    B00 = P
    B10 = (mu_i * P - sigma * Su_rSv)
    B01 = (mu_j * P - sigma * Sv_rSu)

    B20 = ((mu_i * mu_i + sigma2) * P
           - 2.0 * mu_i * sigma * Su_rSv
           + sigma2 * (dSu_du + 2.0 * rho * dSu_dv + rho * rho * dSv_dv))

    B02 = ((mu_j * mu_j + sigma2) * P
           - 2.0 * mu_j * sigma * Sv_rSu
           + sigma2 * (dSv_dv + 2.0 * rho * dSu_dv + rho * rho * dSu_du))

    # NOTE the +rho*sigma2*P term in I11
    B11 = ((mu_i * mu_j + rho * sigma2) * P
           - mu_j * sigma * Su_rSv
           - mu_i * sigma * Sv_rSu
           + sigma2 * (rho * dSu_du + (1.0 + rho * rho) * dSu_dv + rho * dSv_dv))

    return B00, B10, B01, B20, B02, B11

# ---------- Public API ----------
def gaussian_rect_integrals_up_to2_ultra(alpha, q, b, a_ij, a_ji, L_i, L_j):
    """
    Fast evaluation of I_{k,l} (k+l<=2) on [-L_i/2,L_i/2]×[-L_j/2,L_j/2].
    Assumes α>0, |b|<1, finite L_i,L_j.

    Returns dict {(0,0),(1,0),(0,1),(2,0),(0,2),(1,1)} -> float
    """
    rho = b
    denom = 1.0 - rho * rho
    inv_denom = 1.0 / denom

    # μ and Σ
    mu_i = (a_ij + rho * a_ji) * inv_denom
    mu_j = (rho * a_ij + a_ji) * inv_denom
    sigma2 = inv_denom / (2.0 * alpha)
    sigma  = math.sqrt(sigma2)

    # standardized bounds
    half_i, half_j = 0.5 * L_i, 0.5 * L_j
    u_minus, u_plus = (-half_i - mu_i) / sigma, (half_i - mu_i) / sigma
    v_minus, v_plus = (-half_j - mu_j) / sigma, (half_j - mu_j) / sigma

    # Rectangle probability P with one fast mvnun call
    lower = [u_minus, v_minus]
    upper = [u_plus,  v_plus]
    mean  = [0.0, 0.0]
    cov   = [[1.0, rho], [rho, 1.0]]
    P, info = mvn.mvnun(lower, upper, mean, cov)  # Φ₂ over the box
    # (info==0 means OK; you can assert/ignore for speed)

    # Assemble bracket terms in JIT
    B00, B10, B01, B20, B02, B11 = _assemble_given_P(
        mu_i, mu_j, sigma, sigma2, rho, u_minus, u_plus, v_minus, v_plus, P
    )

    # Global prefactor
    a_dot_mu = a_ij * mu_i + a_ji * mu_j
    K = (math.pi / (alpha * math.sqrt(denom))) * math.exp(-alpha * (q - a_dot_mu))

    return {"I00": K * B00, "I10": K * B10, "I01": K * B01, "I11": K * B11, "I20": K * B20, "I02": K * B02}

In [ ]:
import math
from numba import njit

# -------------------- constants --------------------
SQRT1_2       = 1.0 / math.sqrt(2.0)
INV_SQRT_2PI  = 1.0 / math.sqrt(2.0 * math.pi)  # 1/sqrt(2π)
INV_2PI       = 1.0 / (2.0 * math.pi)

# Gauss–Legendre nodes/weights on [-1,1]
_GLN = 32
_GLX, _GLW = np.polynomial.legendre.leggauss(_GLN)
_GLX = _GLX.astype(np.float64)
_GLW = _GLW.astype(np.float64)

# -------------------- JIT special functions --------------------
@njit(fastmath=True, cache=True)
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x * SQRT1_2))

@njit(fastmath=True, cache=True)
def norm_pdf(x):
    return INV_SQRT_2PI * math.exp(-0.5 * x * x)

@njit(fastmath=True, cache=True)
def norm_pdf2(x, y, rho):
    t = 1.0 - rho * rho
    z = (x * x - 2.0 * rho * x * y + y * y) / (2.0 * t)
    return (INV_2PI / math.sqrt(t)) * math.exp(-z)

# -------------------- BvN CDF via Genz integral --------------------
# Φ2(h,k;ρ) = Φ(h)Φ(k) + (1/(2π)) ∫_{0}^{asin(ρ)} exp(-(h^2 - 2hk sinθ + k^2)/(2 cos^2θ)) dθ
@njit(fastmath=True, cache=True)
def _bvn_cdf(h, k, rho, GLX, GLW):
    a = math.asin(rho)  # α
    if a == 0.0:
        return norm_cdf(h) * norm_cdf(k)

    half = 0.5 * a
    s = 0.0
    # Gauss–Legendre: θ_i = half * (x_i + 1),  ∫₀^α f ≈ (α/2) Σ w_i f(θ_i)
    for i in range(GLX.size):
        xi = GLX[i]
        wi = GLW[i]
        th = half * (xi + 1.0)
        c = math.cos(th)
        st = math.sin(th)
        z = (h*h - 2.0*h*k*st + k*k) / (2.0 * c * c)
        s += wi * math.exp(-z)
    return norm_cdf(h) * norm_cdf(k) + (INV_2PI * half) * s

@njit(fastmath=True, cache=True)
def _rect_prob(u_minus, u_plus, v_minus, v_plus, rho, GLX, GLW):
    # Rectangle mass P = Φ2(u+,v+)-Φ2(u-,v+) -Φ2(u+,v-) +Φ2(u-,v-)
    pp = _bvn_cdf(u_plus,  v_plus,  rho, GLX, GLW)
    mp = _bvn_cdf(u_minus, v_plus,  rho, GLX, GLW)
    pm = _bvn_cdf(u_plus,  v_minus, rho, GLX, GLW)
    mm = _bvn_cdf(u_minus, v_minus, rho, GLX, GLW)
    return pp - mp - pm + mm

# -------------------- Boundary assemblers (no numerical diffs) --------------------
@njit(fastmath=True, cache=True)
def _edge_L_of_u(u, v_minus, v_plus, rho):
    tau = math.sqrt(1.0 - rho * rho)
    return norm_cdf((v_plus  - rho*u) / tau) - norm_cdf((v_minus - rho*u) / tau)

@njit(fastmath=True, cache=True)
def _edge_R_of_v(v, u_minus, u_plus, rho):
    tau = math.sqrt(1.0 - rho * rho)
    return norm_cdf((u_plus  - rho*v) / tau) - norm_cdf((u_minus - rho*v) / tau)

@njit(fastmath=True, cache=True)
def _assemble_given_P(mu_i, mu_j, sigma, sigma2, rho,
                      u_minus, u_plus, v_minus, v_plus, P):
    # Edge terms
    L_up = _edge_L_of_u(u_plus,  v_minus, v_plus, rho)
    L_um = _edge_L_of_u(u_minus, v_minus, v_plus, rho)
    R_vp = _edge_R_of_v(v_plus,  u_minus, u_plus, rho)
    R_vm = _edge_R_of_v(v_minus, u_minus, u_plus, rho)

    s_u_plus,  s_u_minus  = norm_pdf(u_plus),  norm_pdf(u_minus)
    s_v_plus,  s_v_minus  = norm_pdf(v_plus),  norm_pdf(v_minus)
    Su = s_u_plus * L_up - s_u_minus * L_um
    Sv = s_v_plus * R_vp - s_v_minus * R_vm

    # Corner PDFs
    c_pp = norm_pdf2(u_plus,  v_plus,  rho)
    c_pm = norm_pdf2(u_plus,  v_minus, rho)
    c_mp = norm_pdf2(u_minus, v_plus,  rho)
    c_mm = norm_pdf2(u_minus, v_minus, rho)

    # Explicit “derivatives” (via PDFs)
    Du_plus, Du_minus = c_pp - c_pm, c_mp - c_mm
    Dv_plus, Dv_minus = c_pp - c_mp, c_pm - c_mm

    dSu_du = (-u_plus  * s_u_plus  * L_up  - rho * Du_plus) \
           + ( u_minus * s_u_minus * L_um  + rho * Du_minus)

    dSv_dv = (-v_plus  * s_v_plus  * R_vp  - rho * Dv_plus) \
           + ( v_minus * s_v_minus * R_vm  + rho * Dv_minus)

    dSu_dv =  c_pp - c_pm - c_mp + c_mm

    # Combine
    Su_rSv = Su + rho * Sv
    Sv_rSu = Sv + rho * Su

    B00 = P
    B10 = (mu_i * P - sigma * Su_rSv)
    B01 = (mu_j * P - sigma * Sv_rSu)
    B20 = ((mu_i * mu_i + sigma2) * P
           - 2.0 * mu_i * sigma * Su_rSv
           + sigma2 * (dSu_du + 2.0 * rho * dSu_dv + rho * rho * dSv_dv))
    B02 = ((mu_j * mu_j + sigma2) * P
           - 2.0 * mu_j * sigma * Sv_rSu
           + sigma2 * (dSv_dv + 2.0 * rho * dSu_dv + rho * rho * dSu_du))
    # note the + rho*sigma^2*P term
    B11 = ((mu_i * mu_j + rho * sigma2) * P
           - mu_j * sigma * Su_rSv
           - mu_i * sigma * Sv_rSu
           + sigma2 * (rho * dSu_du + (1.0 + rho * rho) * dSu_dv + rho * dSv_dv))
    return B00, B10, B01, B20, B02, B11

# -------------------- Public API --------------------
def gaussian_rect_integrals_up_to2_numba(alpha, q, b, a_ij, a_ji, L_i, L_j):
    """
    Computes I_{k,l} (k+l <= 2) on [-L_i/2,L_i/2]×[-L_j/2,L_j/2]
    using a SciPy-free, Numba-JIT implementation.
    Assumes α>0, |b|<1, finite L_i, L_j.
    Returns dict with keys: (0,0),(1,0),(0,1),(2,0),(0,2),(1,1).
    """
    rho   = b
    denom = 1.0 - rho * rho
    inv_d = 1.0 / denom

    mu_i   = (a_ij + rho * a_ji) * inv_d
    mu_j   = (rho * a_ij + a_ji) * inv_d
    sigma2 = inv_d / (2.0 * alpha)
    sigma  = math.sqrt(sigma2)

    # Standardized bounds
    half_i, half_j = 0.5 * L_i, 0.5 * L_j
    u_minus, u_plus = (-half_i - mu_i) / sigma, (half_i - mu_i) / sigma
    v_minus, v_plus = (-half_j - mu_j) / sigma, (half_j - mu_j) / sigma

    # Rectangle mass with our JIT Φ2
    P = _rect_prob(u_minus, u_plus, v_minus, v_plus, rho, _GLX, _GLW)

    # Boundary assembly (returns brackets)
    B00, B10, B01, B20, B02, B11 = _assemble_given_P(
        mu_i, mu_j, sigma, sigma2, rho, u_minus, u_plus, v_minus, v_plus, P
    )

    # Global K
    a_dot_mu = a_ij * mu_i + a_ji * mu_j
    K = (math.pi / (alpha * math.sqrt(denom))) * math.exp(-alpha * (q - a_dot_mu))

    return {"I00": K * B00, "I10": K * B10, "I01": K * B01, "I11": K * B11, "I20": K * B20, "I02": K * B02}


In [20]:
def compute_Ik_l_all_with_quad(p: Params) -> Dict[str, float]:
    alpha, r, a_ij, a_ji, b, L_i, L_j = p.a, p.r, -p.b, -p.c, p.d, p.L_i, p.L_j
    if not (abs(b) <= 1):
        raise ValueError("Require |b|<1 for positive-definite quadratic form.")
    if alpha <= 0 or r < 0 or L_i <= 0 or L_j <= 0:
        raise ValueError("Require alpha>0, r>0, L_i>0, L_j>0.")

    q00 = fast_zrl_src_kl(L_i, L_j, r*r,r*a_ij,r*a_ji, b, alpha*2, 1., k=0, l=0)
    q10 = fast_zrl_src_kl(L_j, L_i, r*r,r*a_ji,r*a_ij, b, alpha*2, 1., k=0, l=1)
    q01 = fast_zrl_src_kl(L_i, L_j, r*r,r*a_ij,r*a_ji, b, alpha*2, 1., k=0, l=1)
    q11 = fast_zrl_src_kl(L_i, L_j, r*r,r*a_ij,r*a_ji, b, alpha*2, 1., k=1, l=1)
    q20 = fast_zrl_src_kl(L_j, L_i, r*r,r*a_ji,r*a_ij, b, alpha*2, 1., k=0, l=2)
    q02 = fast_zrl_src_kl(L_i, L_j, r*r,r*a_ij,r*a_ji, b, alpha*2, 1., k=0, l=2)
    return {
        "q00": q00,
        "q10": q10,
        "q01": q01,
        "q11": q11,
        "q20": q20,
        "q02": q02,
    }

In [37]:
a = 1.
p = Params(a=a, r=1.0, b=0.8, c=-0.4, d=0.3, L_i=300.0, L_j=200.5)
p_map = ParamsMapped(
    a=p.a,
    q=p.r**2,
    a_ij=-p.b * p.r,
    a_ji=-p.c * p.r,
    d=p.d,
    L_i=p.L_i,
    L_j=p.L_j
)
# Profile the three implementations for speed
n_iter = 10000

# start = time.time()
# for _ in tqdm(range(n_iter)):
#     vals = compute_Ik_l_all(p)
# end = time.time()
# print(f"Old formula time: {end - start:.4f} sec")

# start = time.time()
# for _ in tqdm(range(n_iter)):
#     p.a += .0001
#     q_vals = compute_Ik_l_all_with_quad(p)
# end = time.time()
# print(f"Quad formula time: {end - start:.4f} sec")
# p.a = a

# start = time.time()
# for _ in tqdm(range(n_iter)):
#     p.a += .0001
#     aij_vals = gaussian_rect_integrals_up_to2_compact(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
# end = time.time()
# print(f"Compact formula time: {end - start:.4f} sec")
# p.a = 1.7

start = time.time()
for _ in tqdm(range(n_iter)):
    p.a += .0001
    aij_vals = gaussian_rect_integrals_up_to2_ultra(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
end = time.time()
print(f"Ultra formula time: {end - start:.4f} sec")
p.a = a

start = time.time()
for _ in tqdm(range(n_iter)):
    p.a += .0001
    aij_vals = gaussian_rect_integrals_up_to2_numba(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
end = time.time()
print(f"Numba formula time: {end - start:.4f} sec")
p.a = a

start = time.time()
for _ in tqdm(range(n_iter)):
    p.a += .0001
    aij_vals = gaussian_rect_integrals_up_to2_numba_dw(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
end = time.time()
print(f"DW formula time: {end - start:.4f} sec")
p.a = a

q_vals = compute_Ik_l_all_with_quad(p)
aij_vals = gaussian_rect_integrals_up_to2_compact(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
ultra_vals = gaussian_rect_integrals_up_to2_ultra(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
numba_vals = gaussian_rect_integrals_up_to2_numba(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)
dw_vals = gaussian_rect_integrals_up_to2_numba_dw(p.a, p_map.q, p.d, p_map.a_ij, p_map.a_ji, p.L_i, p.L_j)

print(q_vals)
print(aij_vals)
print(ultra_vals)
print(numba_vals)
print(dw_vals)

  0%|          | 0/10000 [00:00<?, ?it/s]/var/folders/st/fgw_z95d55x71m603j17ld300000gr/T/ipykernel_15139/260671678.py:113: DeprecationWarning: `scipy.stats.mvn.mvnun` is deprecated along with the `scipy.stats.mvn` namespace. `scipy.stats.mvn.mvnun` will be removed in SciPy 1.14.0, and the `scipy.stats.mvn` namespace will be removed in SciPy 2.0.0.
  P, info = mvn.mvnun(lower, upper, mean, cov)  # Φ₂ over the box
100%|██████████| 10000/10000 [00:00<00:00, 70665.90it/s]


Ultra formula time: 0.1441 sec


100%|██████████| 10000/10000 [00:00<00:00, 216432.17it/s]


Numba formula time: 0.0483 sec


100%|██████████| 10000/10000 [00:00<00:00, 287324.39it/s]

DW formula time: 0.0371 sec
{'q00': 2.3632010892509103, 'q10': -1.7659085062534277, 'q01': 0.41550788382433596, 'q11': 0.07904923339240535, 'q20': 2.618042119645919, 'q02': 1.371518468172911}
{'I00': 2.36320108925091, 'I10': -1.7659085062534272, 'I01': 0.41550788382433584, 'I11': 0.07904923339240441, 'I20': 2.618042119645918, 'I02': 1.3715184681729105}
{'I00': 2.36320108925091, 'I10': -1.7659085062534272, 'I01': 0.41550788382433584, 'I11': 0.07904923339240442, 'I20': 2.6180421196459176, 'I02': 1.3715184681729105}
{'I00': 2.36320108925091, 'I10': -1.7659085062534272, 'I01': 0.41550788382433584, 'I11': 0.07904923339240442, 'I20': 2.6180421196459176, 'I02': 1.3715184681729105}
{'I00': 2.36320108925091, 'I10': -1.7659085062534272, 'I01': 0.41550788382433584, 'I11': 0.07904923339240442, 'I20': 2.6180421196459176, 'I02': 1.3715184681729105}
